In [28]:
import os

os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages "
    "org.apache.hadoop:hadoop-aws:3.3.4,"
    "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
    "org.postgresql:postgresql:42.7.3 "
    "pyspark-shell"
)


from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("chartevents-session") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "com.amazonaws.auth.DefaultAWSCredentialsProviderChain") \
    .getOrCreate()




In [29]:
df = spark.read.text("s3a://kafka-consumer-spark/consumer/chartevents/")
df.show()

+--------------------+
|               value|
+--------------------+
|[{"row_id": 86138...|
|[{"row_id": 23537...|
|[{"row_id": 67972...|
|[{"row_id": 23537...|
|[{"row_id": 27285...|
|[{"row_id": 86138...|
|[{"row_id": 27285...|
|[{"row_id": 27285...|
|[{"row_id": 27423...|
|[{"row_id": 27423...|
|[{"row_id": 27285...|
|[{"row_id": 27285...|
|[{"row_id": 27423...|
|[{"row_id": 27285...|
|[{"row_id": 73470...|
|[{"row_id": 27285...|
|[{"row_id": 27285...|
|[{"row_id": 27423...|
|[{"row_id": 27423...|
|[{"row_id": 27285...|
+--------------------+
only showing top 20 rows



In [30]:
df_test = spark.read \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://3.94.79.85:5432/mimic") \
    .option("dbtable", "chartevents") \
    .option("user", "postgres") \
    .option("password", "123456") \
    .option("driver", "org.postgresql.Driver") \
    .load()

df_test.show(5)

+--------+----------+-------+----------+------+----------+----------+-----+---------------+--------------------+--------+-------+-----+------------+--------+-----------------+
|  row_id|subject_id|hadm_id|icustay_id|itemid| charttime| storetime| cgid|          value|            valuenum|valueuom|warning|error|resultstatus| stopped|       item_label|
+--------+----------+-------+----------+------+----------+----------+-----+---------------+--------------------+--------+-------+-----+------------+--------+-----------------+
|86138412|     10102| 164869|    223870|   456|2105-06-08|2105-06-08|18784|             64|64.00000000000000...|    mmHg|      0|    0|  processing|NotStopd|         NBP Mean|
|86138413|     10102| 164869|    223870|   522|2105-06-08|2105-06-08|18784|        At Rest|-100.000000000000...|     Non|      0|    0|  processing|NotStopd|       Pain Cause|
|86138414|     10102| 164869|    223870|   524|2105-06-08|2105-06-08|18784|6-Mod to Severe|-100.000000000000...|     Non

In [31]:
d_items = spark.read \
    .option("header", True) \
    .csv("s3a://aws-glue-trying/SourceData/mimic_Demo/D_ITEMS.csv")

In [32]:
from pyspark.sql.functions import col

d_items = d_items.withColumn(
    "itemid",
    col("itemid").cast("int")
)

In [33]:
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType

schema = StructType([
    StructField("cgid", LongType(), True),
    StructField("charttime", StringType(), True),
    StructField("error", StringType(), True),
    StructField("hadm_id", LongType(), True),
    StructField("icustay_id", LongType(), True),
    StructField("itemid", LongType(), True),
    StructField("resultstatus", StringType(), True),
    StructField("row_id", LongType(), True),
    StructField("stopped", StringType(), True),
    StructField("storetime", StringType(), True),
    StructField("subject_id", LongType(), True),
    StructField("value", StringType(), True),
    StructField("valuenum", DoubleType(), True),
    StructField("valueuom", StringType(), True),
    StructField("warning", StringType(), True)
])

In [34]:
df_one = spark.readStream \
    .schema(schema) \
    .format("json") \
    .option("maxFilesPerTrigger", 1) \
    .load("s3a://kafka-consumer-spark/consumer/chartevents/")

In [38]:
from pyspark.sql.functions import col, coalesce, lit, to_timestamp

In [41]:
def process_batch(df, batch_id):
        
    df_clean = df \
        .withColumn("charttime", to_timestamp(col("charttime"))) \
        .withColumn("storetime", to_timestamp(col("storetime")))
    df_clean = df_clean \
        .withColumn("warning", col("warning").cast("int")) \
        .withColumn("error", col("error").cast("int"))
    df_clean = df_clean.select(
        col("row_id").cast("int").alias("ROW_ID"),
        col("subject_id").cast("int").alias("SUBJECT_ID"),
        col("hadm_id").cast("int").alias("HADM_ID"),
        col("icustay_id").cast("int").alias("ICUSTAY_ID"),
        col("itemid").cast("int").alias("ITEMID"),
        col("charttime").alias("CHARTTIME"),
        col("storetime").alias("STORETIME"),
        col("cgid").cast("int").alias("CGID"),
        col("value").alias("VALUE"),
        col("valuenum").alias("VALUENUM"),
        col("valueuom").alias("VALUEUOM"),
        col("warning").alias("WARNING"),
        col("error").alias("ERROR"),
        col("resultstatus").alias("RESULTSTATUS"),
        col("stopped").alias("STOPPED")
    )
    
    df_clean = df_clean.fillna({
        "warning": 0,
        "error": 0,
        "resultstatus": "processing",
        "stopped": "NotStopped",
        "valuenum": -100.0,
        "valueuom": "Non"
    })
    
    df_joined = (
        df_clean.join(
            d_items.select(
                col("itemid"),
                col("label").alias("item_label")
            ),
            on="itemid",
            how="left"
        )
        .withColumn(
            "item_label",
            coalesce(col("item_label"), lit("Item"))
        )
    )
    
    df_joined.write \
        .format("jdbc") \
        .option("url", "jdbc:postgresql://3.94.79.85:5432/mimic") \
        .option("dbtable", "chartevents") \
        .option("user", "postgres") \
        .option("password", "123456") \
        .option("driver", "org.postgresql.Driver") \
        .mode("append") \
        .save()

In [42]:
query = df_one.writeStream \
    .trigger(processingTime="5 seconds") \
    .foreachBatch(process_batch) \
    .option("checkpointLocation", "checkpointchart") \
    .start()

query.awaitTermination()

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/socket.py", line 718, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [43]:
query.lastProgress["sources"]

[{'description': 'FileStreamSource[s3a://kafka-consumer-spark/consumer/chartevents]',
  'startOffset': {'logOffset': 79},
  'endOffset': {'logOffset': 79},
  'latestOffset': None,
  'numInputRows': 0,
  'inputRowsPerSecond': 0.0,
  'processedRowsPerSecond': 0.0}]

In [44]:
spark.stop()